[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 02](README.md)

# C++20: RAII, jthread y atomics

**Tema:** 02 · **Sesiones:** 9, 10 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Cuándo una operación atómica es suficiente y qué relación de memoria necesita el algoritmo?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** RAII resuelve vida útil; los atomics resuelven operaciones y órdenes concretos. Elegirlos bien requiere decir qué dato se publica y quién debe observarlo.

**Prerrequisitos.**

- Partición de datos y referencia serial del tema 01.
- Punteros, funciones y vida útil de objetos en C/C++.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Diferenciar atomicidad, orden y exclusión mutua.
- Usar RAII y `std::jthread` para administrar vida útil.
- Elegir órdenes de memoria a partir del protocolo, no por intuición.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

Una variable atómica evita carreras sobre esa variable, pero no vuelve atómico un invariante compuesto.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

Release publica escrituras anteriores y acquire permite observarlas cuando lee el valor publicado.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

`memory_order_relaxed` sirve para contadores sin relación de publicación; seq_cst ofrece el modelo global más fuerte y costoso.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- carrera — accesos concurrentes incompatibles sin orden suficiente
- happens-before — relación que hace visible un efecto entre hilos
- deadlock — ciclo de espera que impide el progreso


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Happens Before

![Publicación de datos entre productor y consumidor](../../images/happens-before.svg)

**Cómo leerlo.** La flecha central representa sincronización, no el mero paso del tiempo. Sin esa relación el consumidor no tiene garantía de observar la escritura.

### Fork Join

![Región serial que crea y reúne trabajadores](../../images/fork-join.svg)

**Cómo leerlo.** La región posterior al join solo puede consumir el resultado cuando todos los trabajadores necesarios terminaron y publicaron sus parciales.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "02"
NOTEBOOK = "02_memoria_compartida/03_cpp20_atomics.ipynb"
assert (ROOT / "curso" / "notebooks" / "02_memoria_compartida" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Separación de contadores

**Situación.** Se calcula padding para evitar compartir líneas de caché cuando cada hilo escribe su contador.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
def padded_stride(value_bytes, cache_line=64):
    return ((value_bytes + cache_line - 1) // cache_line) * cache_line
assert padded_stride(8) == 64
assert padded_stride(72) == 128
for size in (4, 8, 16, 64, 72): print(size, padded_stride(size))


### Explicación del resultado

En C++ se prefiere `std::hardware_destructive_interference_size` cuando está disponible, verificando la implementación.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Selección razonada

**Situación.** Una tabla relaciona patrones mínimos con el orden que debe justificarse.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
protocols = {
    "contador independiente": ("relaxed", "no publica otros datos"),
    "bandera de publicación": ("release/acquire", "publica y consume estado previo"),
    "algoritmo sin prueba formal": ("seq_cst", "punto de partida conservador"),
    "invariante compuesto": ("mutex", "varias ubicaciones cambian juntas"),
}
assert protocols["invariante compuesto"][0] == "mutex"
for pattern, decision in protocols.items(): print(f"{pattern:27} -> {decision[0]:15} | {decision[1]}")


### Lectura razonada

La tabla no reemplaza la prueba de happens-before del algoritmo concreto.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Cuándo `memory_order_relaxed` es suficiente y qué garantía deliberadamente no ofrece?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Reescribir un contador con relaxed y justificar por qué no publica datos.
2. Modelar una bandera release/acquire con productor y consumidor.
3. Ejecutar sanitizadores y comparar con una versión protegida por mutex.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Usar `volatile` como sincronización.
- Aplicar relaxed a una publicación sin relación de memoria.
- Crear hilos sin una política clara de cancelación y join.


## Criterios de aceptación

- No hay carreras en el detector.
- Cada orden de memoria tiene justificación escrita.
- La referencia protegida por mutex produce el mismo resultado.


## Síntesis

- La pregunta que debes poder responder es: **¿Cuándo una operación atómica es suficiente y qué relación de memoria necesita el algoritmo?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Ejemplos Pthreads relacionados](../../../pthreads/)
- [Planeación C++20](../../../docs/PLANEACION_CURSO.md)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 02](README.md)
